# Structured Outputs on Amazon Bedrock

A walkthrough of how to get JSON-schema-constrained output from models on Amazon Bedrock, across both Bedrock endpoints (`bedrock-runtime` and `bedrock-mantle`) and the mechanisms available on each.

## Background

Anthropic's own [structured outputs documentation](https://platform.claude.com/docs/en/build-with-claude/structured-outputs) states that `output_config` (Anthropic's native JSON-schema-constrained output field) was, at launch, only available on Bedrock through a legacy integration, for a short list of models (Claude Opus 4.6, Sonnet 4.6, Sonnet 4.5, Opus 4.5, Haiku 4.5) -- not through Converse at all.

That picture has moved on. This notebook demonstrates **two independent mechanisms** that work much more broadly today:

1. **Bedrock's own schema field** on Converse -- works across model providers (Anthropic, OpenAI, and others hosted on Bedrock), not just Anthropic.
2. **Anthropic's native `output_config` field** -- still Anthropic-only, but now usable directly on Converse and InvokeModel for current-generation models, not just the legacy-list ones.

Plus a look at the OpenAI-compatible **Responses API**, which supports structured output identically on both Bedrock endpoints.

> **Prerequisites:** AWS credentials with Bedrock access (`aws configure` or any standard credential provider), and `pip install -r requirements.txt`.

## 1. Converse API -- Bedrock's own schema field (cross-provider)

Pass `outputConfig.textFormat` directly on `converse()`. This has been verified working across Anthropic, OpenAI, and Moonshot (Kimi) models hosted on Bedrock -- it's a platform-level feature, not an Anthropic-specific one.

In [1]:
import json
import boto3

REGION = "us-east-1"
MODEL_ID = "us.anthropic.claude-sonnet-5-5"  # swap for any Converse-supported model
PROMPT = "Extract the person from: John Smith is 42 years old."
SCHEMA = {
    "type": "object",
    "properties": {"name": {"type": "string"}, "age": {"type": "integer"}},
    "required": ["name", "age"],
    "additionalProperties": False,
}

client = boto3.client("bedrock-runtime", region_name=REGION)

response = client.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": PROMPT}]}],
    outputConfig={
        "textFormat": {
            "type": "json_schema",
            "structure": {"jsonSchema": {"schema": json.dumps(SCHEMA), "name": "person"}},
        }
    },
)

text = response["output"]["message"]["content"][0]["text"]
print(text)
print(json.loads(text))

{"name":"John Smith","age":42}
{'name': 'John Smith', 'age': 42}


### Does it really enforce the schema?

Two quick checks worth knowing about, since "it returned valid JSON" alone doesn't prove much for an easy prompt:

- **Off-topic prompt test**: ask something unrelated ("what's the capital of France?") under this same person schema. A genuinely constrained model is forced into something like `{"name": "Paris", "age": 0}` instead of answering in prose.
- **Broken schema test**: pass an invalid JSON Schema string. If the field is real, Bedrock rejects it server-side with a validation error instead of silently ignoring it.

Both checks pass for this field -- it's real grammar-level constrained decoding, not a loose "please reply in JSON" hint.

## 2. Converse / InvokeModel -- Anthropic's native `output_config` field

This is the field documented on [platform.claude.com](https://platform.claude.com/docs/en/build-with-claude/structured-outputs). It's Anthropic-only -- other providers either ignore it silently or reject it outright. On Converse it has to be tunneled through `additionalModelRequestFields`; on InvokeModel it's just a top-level field in the Anthropic Messages body.

In [2]:
# Converse + Anthropic-native output_config
response = client.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": PROMPT}]}],
    additionalModelRequestFields={
        "output_config": {"format": {"type": "json_schema", "schema": SCHEMA}}
    },
)

text = response["output"]["message"]["content"][0]["text"]
print(text)
print(json.loads(text))

{"age":42,"name":"John Smith"}
{'age': 42, 'name': 'John Smith'}


In [3]:
# InvokeModel + Anthropic-native output_config (same field, Anthropic Messages body shape)
body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 64,
    "messages": [{"role": "user", "content": PROMPT}],
    "output_config": {"format": {"type": "json_schema", "schema": SCHEMA}},
}

response = client.invoke_model(modelId=MODEL_ID, body=json.dumps(body))
result = json.loads(response["body"].read())

text = result["content"][0]["text"]
print(text)
print(json.loads(text))

{"name":"John Smith","age":42}
{'name': 'John Smith', 'age': 42}


## 3. Responses API -- same code, works on both endpoints

The OpenAI-compatible Responses API uses `text.format.type = "json_schema"` with `strict: true`. Auth is a short-lived bearer token derived from your AWS credentials, not SigV4 request signing, so you can point the plain `openai` SDK straight at either Bedrock endpoint via `base_url`.

Model id format differs slightly by endpoint: `bedrock-mantle` takes the plain model id (`openai.gpt-5.6-sol`), while `bedrock-runtime` needs the cross-region profile id (`us.openai.gpt-5.6-sol`) -- the same requirement Converse has, not a Responses-API-specific quirk.

In [4]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

token = provide_token(REGION)

text_format = {
    "format": {"type": "json_schema", "name": "person", "schema": SCHEMA, "strict": True}
}

mantle_client = OpenAI(base_url=f"https://bedrock-mantle.{REGION}.api.aws/openai/v1", api_key=token)
mantle_resp = mantle_client.responses.create(model="openai.gpt-5.6-sol", input=PROMPT, text=text_format)
print("bedrock-mantle Responses API:", mantle_resp.output_text)

runtime_client = OpenAI(base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1", api_key=token)
runtime_resp = runtime_client.responses.create(model="us.openai.gpt-5.6-sol", input=PROMPT, text=text_format)
print("bedrock-runtime Responses API:", runtime_resp.output_text)

bedrock-mantle Responses API: {"name":"John Smith","age":42}


bedrock-runtime Responses API: {"name":"John Smith","age":42}


## Summary

| Mechanism | API | Endpoint(s) | Provider scope |
|---|---|---|---|
| Bedrock-native `outputConfig` | Converse | `bedrock-runtime` | Cross-provider (Anthropic, OpenAI, Moonshot verified) |
| Anthropic-native `output_config` | Converse (`additionalModelRequestFields`) | `bedrock-runtime` | Anthropic only |
| Anthropic-native `output_config` | InvokeModel | `bedrock-runtime` | Anthropic only |
| `text.format` (`json_schema`, `strict`) | Responses API | `bedrock-runtime` **and** `bedrock-mantle` | Whatever models that endpoint hosts |

A couple of practical notes:
- When the response is cut off by `max_tokens`, `stopReason`/`stop_reason` comes back as `max_tokens` and the JSON is left incomplete -- the schema guarantee only holds for a completed turn, same as Anthropic's native behavior.
- Key order in the returned JSON isn't guaranteed to match the schema's property order -- harmless, just don't rely on it positionally.

## Notes
This project is for educational purposes and is not designed for production use. For production use, please implement proper security, scalability, and compliance measures appropriate for your use case.